# Phase 6 Stage 0 - Environment and Data Readiness

Notebook ini mengunci kesiapan environment dan target data sebelum implementasi IndoBERTweet, Temporal LSTM, dan GraphSAGE. Stage 0 tidak mengubah data raw atau label manual.

**Status:** baseline proxy tetap dipertahankan; deep-learning dependencies dicek; kebutuhan data dihitung dari artifact canonical.

In [1]:
from pathlib import Path
import importlib.util
import json
import platform
import sys
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'PRD.md').exists())
sys.path.insert(0, str(ROOT / 'src'))
from models.phase6_advanced import MODEL_FEATURES

DEPENDENCIES = ('torch', 'transformers', 'torch_geometric', 'accelerate', 'sentencepiece', 'safetensors')
dependency_status = {name: importlib.util.find_spec(name) is not None for name in DEPENDENCIES}
print('Repository:', ROOT)
print('Python:', platform.python_version())
print('Dependency status:', dependency_status)
print('Stage 0 setup: OK')

Repository: /home/andry-alfisi/Project/Project/Buzzer-Detection-
Python: 3.13.9
Dependency status: {'torch': True, 'transformers': True, 'torch_geometric': True, 'accelerate': True, 'sentencepiece': True, 'safetensors': True}
Stage 0 setup: OK


In [2]:
def read_csv(path):
    return pd.read_csv(path, dtype='string', keep_default_na=True)

inventory = []
total_labeled_multi_post = 0
total_valid_edges = 0
for batch in ('batch2', 'batch3'):
    posts = read_csv(ROOT / 'data' / 'features' / f'{batch}_posts_with_features.csv')
    accounts = read_csv(ROOT / 'data' / 'features' / f'{batch}_account_features.csv')
    labels = read_csv(ROOT / 'data' / 'labels' / f'{batch}_labeled_canonical.csv')
    interactions = read_csv(ROOT / 'data' / 'features' / f'{batch}_interactions.csv')
    post_counts = posts.groupby('account_id').size()
    valid_edges = interactions[interactions['is_valid_edge'].eq('True')]
    labeled_multi_post = int(labels['account_id'].isin(post_counts[post_counts >= 2].index).sum())
    total_labeled_multi_post += labeled_multi_post
    total_valid_edges += len(valid_edges)
    inventory.append({
        'batch': batch,
        'posts': len(posts),
        'accounts': len(accounts),
        'labels': len(labels),
        'organic_labels': int((labels['label'] == '0').sum()),
        'buzzer_labels': int((labels['label'] == '1').sum()),
        'multi_post_accounts': int((post_counts >= 2).sum()),
        'labeled_multi_post': labeled_multi_post,
        'valid_edges': len(valid_edges),
    })

inventory_df = pd.DataFrame(inventory)
print(inventory_df.to_string(index=False))
print('Current totals:', inventory_df.sum(numeric_only=True).to_dict())
assert all(column in read_csv(ROOT / 'data' / 'features' / f'{batch}_account_features.csv').columns for column in ('account_id', 'issue_id') for batch in ('batch2', 'batch3'))
print('Canonical data contract: OK')

 batch  posts  accounts  labels  organic_labels  buzzer_labels  multi_post_accounts  labeled_multi_post  valid_edges
batch2   1514      1007      50              27             23                  168                  10          651
batch3   1081       796      50              21             29                  134                  10          714
Current totals: {'posts': 2595, 'accounts': 1803, 'labels': 100, 'organic_labels': 48, 'buzzer_labels': 52, 'multi_post_accounts': 302, 'labeled_multi_post': 20, 'valid_edges': 1365}
Canonical data contract: OK


In [3]:
TARGETS = {
    'minimum_prototype': {
        'labeled_accounts': 300,
        'organic_labels': 150,
        'buzzer_labels': 150,
        'labeled_multi_post': 100,
        'valid_edges': 1500,
    },
    'recommended_stable': {
        'labeled_accounts': 600,
        'organic_labels': 300,
        'buzzer_labels': 300,
        'labeled_multi_post': 200,
        'valid_edges': 3000,
    },
}

current = {
    'labeled_accounts': int(inventory_df['labels'].sum()),
    'organic_labels': int(inventory_df['organic_labels'].sum()),
    'buzzer_labels': int(inventory_df['buzzer_labels'].sum()),
    'labeled_multi_post': int(total_labeled_multi_post),
    'valid_edges': int(total_valid_edges),
}

gap_rows = []
for target_name, target in TARGETS.items():
    gap_rows.append({
        'target': target_name,
        **{key: max(0, value - current[key]) for key, value in target.items()},
    })
gap_df = pd.DataFrame(gap_rows)
print('Current:', current)
print('Additional data needed:')
print(gap_df.to_string(index=False))

Current: {'labeled_accounts': 100, 'organic_labels': 48, 'buzzer_labels': 52, 'labeled_multi_post': 20, 'valid_edges': 1365}
Additional data needed:
            target  labeled_accounts  organic_labels  buzzer_labels  labeled_multi_post  valid_edges
 minimum_prototype               200             102             98                  80          135
recommended_stable               500             252            248                 180         1635


In [4]:
readiness = {
    'status': 'READY_FOR_DATA_COLLECTION_AND_ALGORITHM_IMPLEMENTATION',
    'python_version': platform.python_version(),
    'dependencies': dependency_status,
    'current': current,
    'targets': TARGETS,
    'model_features': MODEL_FEATURES,
    'constraints': [
        'Do not modify data/raw.',
        'Do not use buzzer_score as a feature or ground truth.',
        'Keep account_id, post_id, and edge IDs as strings.',
        'Keep labels balanced by issue and include multi-post labeled accounts.',
        'Use group, temporal, and cross-issue evaluation.',
    ],
}

output_dir = ROOT / 'output' / 'phase6_stage0'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'stage0_readiness.json'
output_path.write_text(json.dumps(readiness, indent=2, default=str), encoding='utf-8')
print(json.dumps(readiness, indent=2, default=str))
print('Stage 0 readiness written to:', output_path)

{
  "status": "READY_FOR_DATA_COLLECTION_AND_ALGORITHM_IMPLEMENTATION",
  "python_version": "3.13.9",
  "dependencies": {
    "torch": true,
    "transformers": true,
    "torch_geometric": true,
    "accelerate": true,
    "sentencepiece": true,
    "safetensors": true
  },
  "current": {
    "labeled_accounts": 100,
    "organic_labels": 48,
    "buzzer_labels": 52,
    "labeled_multi_post": 20,
    "valid_edges": 1365
  },
  "targets": {
    "minimum_prototype": {
      "labeled_accounts": 300,
      "organic_labels": 150,
      "buzzer_labels": 150,
      "labeled_multi_post": 100,
      "valid_edges": 1500
    },
    "recommended_stable": {
      "labeled_accounts": 600,
      "organic_labels": 300,
      "buzzer_labels": 300,
      "labeled_multi_post": 200,
      "valid_edges": 3000
    }
  },
  "model_features": [
    "avg_tags",
    "avg_body_len",
    "url_rate",
    "avg_words",
    "n_posts",
    "active_hours",
    "post_rate",
    "n_replies",
    "n_quotes",
    "n_origi

## Keputusan Stage 0

- Training prototype dapat dimulai dengan data sekarang untuk menguji pipeline, tetapi belum layak disebut model final.
- Target minimum pengumpulan: **300 akun berlabel seimbang**, **100 akun berlabel multi-post**, dan **1.500 edge valid**.
- Target yang lebih aman untuk perbandingan tiga algoritma: **600 akun berlabel seimbang**, **200 akun berlabel multi-post**, dan **3.000 edge valid**.
- IndoBERTweet dapat dimulai dengan frozen encoder; LSTM dan GraphSAGE perlu tambahan label multi-post dan label pada node yang terhubung graph.